# Building a Coding Agent from Claude API Primitives

This notebook builds a small, working coding agent directly on top of the raw **Claude API** (the `anthropic` Python SDK) — no LangGraph, no LangChain, no agent framework of any kind. The goal is to make the underlying mechanics fully visible: the exact request/response shapes that a tool-using agentic loop is built from. This is the same core loop mechanism — call the model, inspect `stop_reason`, execute tools locally, feed results back, repeat — that products like Claude Code are built on top of (with a great deal of additional engineering: a much larger built-in tool surface, permissions, context compaction, subagents, etc.). Here we implement the loop itself, by hand, so the mechanism is transparent.

## A Deliberate Exception to This Repo's `helpers.get_llm()` Convention

Every other LangGraph-phase notebook in this repository initializes its LLM through `from helpers import get_llm`, a small factory in the repo's `helpers/` package that routes to OpenAI, Groq, or Databricks depending on platform (see `helpers/utils.py` at the repo root). That factory has **no Anthropic/Claude branch** — it does not construct a native `anthropic` client, only `ChatOpenAI` / `ChatGroq` / `ChatDatabricks` / a Databricks AI Gateway wrapper.

This notebook is specifically about **first-party Claude tooling** — the Claude API and the tool-use loop it's built on — so it deliberately bypasses `helpers.get_llm()` and imports the `anthropic` SDK directly:

```python
from anthropic import Anthropic
```

This is a repo-documented exception, not an oversight: teaching first-party Claude API usage requires using Claude's own native SDK, not a LangChain-wrapped chat model.

## Environment Variable Note

This notebook requires **`ANTHROPIC_API_KEY`** to be set (in your `.env` at the project root, or exported in your shell). This variable is not among this repo's currently-documented required environment variables in `CLAUDE.md` (which lists `OPENAI_API_KEY`, `GROQ_API_KEY`, `GOOGLE_API_KEY`, `TAVILY_API_KEY`, and Databricks credentials) — you will need to add `ANTHROPIC_API_KEY=your-key-here` to `.env` specifically to run the cells in this notebook.

## Safety Note — Everything Runs in a Throwaway Sandbox

Every file operation and every piece of code the agent runs in this notebook is confined to a **`tempfile.mkdtemp()` scratch directory** created fresh when the notebook runs, completely outside this git repository. The agent's tools (`list_files`, `read_file`, `write_file`, `run_python`) resolve every path against that sandbox directory and reject anything that would escape it (e.g. `../../etc/passwd`-style traversal). The agent **never reads, writes, or executes anything in the real repository.** `run_python` executes as a subprocess (`subprocess.run`) with a timeout, with its working directory set to the sandbox — not a bare `exec()` in the notebook's own process.

In [ ]:
# ============ SETUP ============
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()  # expects ANTHROPIC_API_KEY in the project-root .env

client = Anthropic()  # reads ANTHROPIC_API_KEY from the environment
MODEL = "claude-opus-5"

# A fresh, throwaway sandbox directory -- created outside the repo, deleted by the OS eventually.
# ALL file/code operations in this notebook are confined to this directory.
SANDBOX_DIR = Path(tempfile.mkdtemp(prefix="claude_coding_agent_"))
print(f"Sandbox directory: {SANDBOX_DIR}")

## 1. The Core Tool-Use Loop

Anthropic's tool-use pattern works like this:

1. You describe your tools with a JSON schema (`name`, `description`, `input_schema`) and pass them via the `tools=` parameter on `client.messages.create(...)`.
2. Claude decides, based on the conversation, whether it needs a tool. If so, the response comes back with `stop_reason == "tool_use"` and one or more `tool_use` content blocks — each with a `name`, an `input` dict (already parsed from JSON), and an `id`.
3. **You** — the client — execute the requested tool(s) locally. Anthropic never runs your code; it only ever asks you to.
4. You send the results back as `tool_result` content blocks in a new `user` message, matching each result's `tool_use_id` to the `id` of the `tool_use` block it answers.
5. Repeat from step 2. The loop ends when Claude replies with `stop_reason == "end_turn"` (a final text answer, no more tool calls).

This request/response cycle is stateless on the server side — the *entire* conversation history (including every past `tool_use` and `tool_result` block) is resent on every turn.

In [ ]:
# ============ TOOL SCHEMAS ============
# Each tool is a plain JSON-schema dict: name, description, input_schema.
# No SDK magic here -- this is exactly the `tools` list documented for `client.messages.create`.

TOOLS = [
    {
        "name": "list_files",
        "description": (
            "List every file currently in the sandbox scratch directory. "
            "Takes no arguments."
        ),
        "input_schema": {
            "type": "object",
            "properties": {},
            "additionalProperties": False,
        },
    },
    {
        "name": "read_file",
        "description": "Read and return the full text contents of a file in the sandbox scratch directory.",
        "input_schema": {
            "type": "object",
            "properties": {
                "filename": {
                    "type": "string",
                    "description": "Filename relative to the sandbox directory, e.g. 'buggy_math.py'.",
                },
            },
            "required": ["filename"],
            "additionalProperties": False,
        },
    },
    {
        "name": "write_file",
        "description": (
            "Create or overwrite a file in the sandbox scratch directory with the given text content."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "filename": {
                    "type": "string",
                    "description": "Filename relative to the sandbox directory.",
                },
                "content": {
                    "type": "string",
                    "description": "Full new text contents of the file (overwrites any existing content).",
                },
            },
            "required": ["filename", "content"],
            "additionalProperties": False,
        },
    },
    {
        "name": "run_python",
        "description": (
            "Execute a Python file that already exists in the sandbox scratch directory, as a "
            "subprocess confined to that directory, and return its stdout/stderr/exit code. "
            "Use this to verify a fix actually works."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "filename": {
                    "type": "string",
                    "description": "Filename of the Python script to run, relative to the sandbox directory.",
                },
            },
            "required": ["filename"],
            "additionalProperties": False,
        },
    },
]

print(f"Defined {len(TOOLS)} tools: {[t['name'] for t in TOOLS]}")

### Tool Execution — Sandboxed and Path-Confined

Each handler below resolves the requested filename against `SANDBOX_DIR` and refuses to touch anything outside it (guarding against `../` traversal or absolute paths). `run_python` shells out via `subprocess.run` with `cwd=SANDBOX_DIR` and a hard timeout — it is never a bare `exec()`/`eval()` in this notebook's own process.

In [ ]:
# ============ SANDBOXED TOOL IMPLEMENTATIONS ============

def _safe_path(filename: str) -> Path:
    """Resolve `filename` against SANDBOX_DIR and refuse any path that escapes it."""
    candidate = (SANDBOX_DIR / filename).resolve()
    sandbox_resolved = SANDBOX_DIR.resolve()
    if sandbox_resolved not in candidate.parents and candidate != sandbox_resolved:
        raise ValueError(
            f"Refusing to access '{filename}': resolves outside the sandbox directory."
        )
    return candidate


def tool_list_files(_input: dict) -> str:
    names = sorted(p.name for p in SANDBOX_DIR.iterdir() if p.is_file())
    return json.dumps(names)


def tool_read_file(tool_input: dict) -> str:
    path = _safe_path(tool_input["filename"])
    if not path.exists():
        raise FileNotFoundError(f"No such file: {tool_input['filename']}")
    return path.read_text()


def tool_write_file(tool_input: dict) -> str:
    path = _safe_path(tool_input["filename"])
    path.write_text(tool_input["content"])
    return f"Wrote {len(tool_input['content'])} characters to {tool_input['filename']}."


def tool_run_python(tool_input: dict) -> str:
    path = _safe_path(tool_input["filename"])
    if not path.exists():
        raise FileNotFoundError(f"No such file: {tool_input['filename']}")
    proc = subprocess.run(
        [sys.executable, str(path)],
        cwd=SANDBOX_DIR,
        capture_output=True,
        text=True,
        timeout=10,
    )
    return json.dumps(
        {"stdout": proc.stdout, "stderr": proc.stderr, "returncode": proc.returncode}
    )


TOOL_IMPLEMENTATIONS = {
    "list_files": tool_list_files,
    "read_file": tool_read_file,
    "write_file": tool_write_file,
    "run_python": tool_run_python,
}


def execute_tool(name: str, tool_input: dict) -> str:
    """Dispatch a tool_use block to its sandboxed implementation. Never raises --
    exceptions are caught by the caller and turned into an `is_error` tool_result."""
    handler = TOOL_IMPLEMENTATIONS.get(name)
    if handler is None:
        raise ValueError(f"Unknown tool: {name}")
    return handler(tool_input)

## 2. The Manual Agentic Loop

This is the client-side loop described above, implemented directly against `client.messages.create(...)`: call Claude, check `stop_reason`, and if it's `"tool_use"`, execute every requested tool locally and send all results back in a single `user` message (Claude may request several tools in parallel in one turn — all their `tool_result` blocks must go back together, not split across messages). The loop prints a full turn-by-turn transcript so the mechanics stay visible.

In [ ]:
# ============ THE MANUAL TOOL-USE LOOP ============

SYSTEM_PROMPT = (
    "You are a careful coding assistant working inside a sandboxed scratch directory. "
    "You have tools to list, read, and write files, and to run a Python file to check your work. "
    "When asked to fix a bug: list the files, read the relevant one, reason about the bug, "
    "write a corrected version of the file, then run it to confirm the fix works before reporting "
    "back to the user."
)


def run_agent(user_message: str, max_iterations: int = 8):
    """Run the manual tool-use loop to completion, printing every turn."""
    messages = [{"role": "user", "content": user_message}]

    print("=" * 78)
    print(f"USER: {user_message}")
    print("=" * 78)

    response = None
    for iteration in range(1, max_iterations + 1):
        response = client.messages.create(
            model=MODEL,
            max_tokens=4096,
            system=SYSTEM_PROMPT,
            tools=TOOLS,
            messages=messages,
        )

        # Show any narration Claude produced alongside/before tool calls.
        for block in response.content:
            if block.type == "text" and block.text.strip():
                print(f"\n[Turn {iteration}] Claude: {block.text.strip()}")

        # Always append the assistant turn -- it must be echoed back verbatim,
        # tool_use blocks included, for the next request to make sense.
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            print(f"\n[Turn {iteration}] stop_reason = {response.stop_reason!r} -- loop ending.")
            break

        tool_use_blocks = [b for b in response.content if b.type == "tool_use"]
        tool_results = []
        for block in tool_use_blocks:
            print(f"\n[Turn {iteration}] TOOL CALL  -> {block.name}({block.input})")
            try:
                result_text = execute_tool(block.name, block.input)
                is_error = False
            except Exception as exc:  # noqa: BLE001 -- surface any failure back to Claude
                result_text = f"Error: {exc}"
                is_error = True
            preview = result_text if len(result_text) <= 400 else result_text[:400] + " ...(truncated)"
            print(f"[Turn {iteration}] TOOL RESULT <- {preview}")

            result_block = {
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": result_text,
            }
            if is_error:
                result_block["is_error"] = True
            tool_results.append(result_block)

        # All tool_result blocks for this turn go back together in one user message.
        messages.append({"role": "user", "content": tool_results})
    else:
        print("\n[!] Reached max_iterations without a final answer.")

    final_text = next(
        (b.text for b in response.content if b.type == "text"), "(no final text produced)"
    )
    print("\n" + "=" * 78)
    print(f"FINAL ANSWER:\n{final_text}")
    print("=" * 78)
    return messages

## 3. Demonstrating Multi-Turn Agentic Behavior

We seed the sandbox with a deliberately-buggy Python file, then ask the agent to find and fix the bug. `cumulative_sum` is supposed to return the running total after each element, but the `range(len(nums) - 1)` loop bound is off by one, so it silently drops the last element. This is the kind of bug that's easy to describe ("there's a bug in this function") but requires reading the code, reasoning about it, and verifying a fix -- exactly what should drive the agent through `list_files` -> `read_file` -> `write_file` -> `run_python`.

In [ ]:
# ============ SEED THE SANDBOX WITH A BUGGY FILE ============

buggy_source = '''\
def cumulative_sum(nums):
    """Return a list where element i holds the sum of nums[0..i] inclusive.

    Example: cumulative_sum([1, 2, 3, 4, 5]) == [1, 3, 6, 10, 15]
    """
    result = []
    total = 0
    for i in range(len(nums) - 1):  # BUG: off-by-one, drops the last element
        total += nums[i]
        result.append(total)
    return result


if __name__ == "__main__":
    print(cumulative_sum([1, 2, 3, 4, 5]))
'''

(SANDBOX_DIR / "buggy_math.py").write_text(buggy_source)
print("Seeded sandbox with buggy_math.py:")
print(buggy_source)

# Show the bug in action before the agent ever sees it
proc = subprocess.run([sys.executable, "buggy_math.py"], cwd=SANDBOX_DIR, capture_output=True, text=True)
print(f"Current (buggy) output: {proc.stdout.strip()}  <- should be [1, 3, 6, 10, 15]")

In [ ]:
# ============ RUN THE AGENT ============

task = (
    "There's a bug in buggy_math.py in this scratch directory -- the cumulative_sum function "
    "isn't returning what its docstring promises. Find the bug, fix it, and verify your fix by "
    "running the file. Then tell me what was wrong."
)

final_messages = run_agent(task)

### Discussion of the Output

Walking through the transcript above:

- **Turn 1** — Claude has no visibility into the sandbox yet, so it calls `list_files` (and typically `read_file` in the same or a following turn) rather than guessing at the file's contents.
- **Tool results come back as plain strings** inside `tool_result` blocks — Claude re-reads the file contents from the conversation, not from any persistent memory of the file system.
- **The fix is applied via `write_file`**, then **verified via `run_python`** before Claude reports back — nothing here forces that order; it emerges from the system prompt's instruction to verify before reporting, combined with the model's own judgment.
- **Every intermediate `assistant` turn (including its `tool_use` blocks) and every `tool_result` turn is resent on the next request** — the API is stateless, so `messages` accumulates the full history including the fix reasoning, the write, and the verification run.
- If you inspect `final_messages`, you'll see the buggy source, Claude's read of it, the corrected `write_file` call, and the `run_python` output confirming `[1, 3, 6, 10, 15]` — the entire mechanism a coding agent runs on, with nothing hidden.

In [ ]:
# ============ VERIFY THE FIX ON DISK (sanity check, outside the agent) ============

fixed_source = (SANDBOX_DIR / "buggy_math.py").read_text()
print(fixed_source)

proc = subprocess.run([sys.executable, "buggy_math.py"], cwd=SANDBOX_DIR, capture_output=True, text=True)
print(f"Output after agent's fix: {proc.stdout.strip()}")

## How This Differs from `AI_Coding_Tool_Landscape/01_AI_Agents_on_the_CLI.ipynb`

A sibling notebook in this phase, `AI_Coding_Tool_Landscape/01_AI_Agents_on_the_CLI.ipynb` (authored separately), surveys the *wider landscape* of CLI coding agents and builds a small, generic toy agent to illustrate the category. **This notebook is different in kind, not just in scope**: it is a from-scratch, deep-dive implementation of *Claude's own native tool-use loop* specifically — the exact request/response contract (`tool_use` / `tool_result` blocks, `stop_reason` semantics, JSON-schema tool definitions) that `client.messages.create(...)` exposes, with nothing abstracted away. Where the landscape notebook answers "what exists and how do these tools compare," this notebook answers "what is actually happening on the wire, turn by turn, when a coding agent calls a tool" — the same mechanism that first-party products like Claude Code are built on top of (plus a much larger toolset, permissions, context management, and other product-level engineering this notebook does not attempt to reproduce).

## Summary — Key Takeaways

- The Claude API's tool-use loop is a simple, fully client-driven cycle: send `tools=` + `messages`, check `stop_reason`, and if it's `"tool_use"`, execute the requested tool(s) yourself and send `tool_result` blocks back — repeat until `stop_reason == "end_turn"`.
- Tools are just JSON schemas (`name`, `description`, `input_schema`) — Claude never runs your code; it only ever asks the client to, via a `tool_use` content block carrying an `id`, a `name`, and a parsed `input` dict.
- Every `tool_result` must reference its `tool_use_id`, and when a single turn requests multiple tools, **all** their results must be returned together in one `user` message.
- The API is stateless — the full conversation, including every past `tool_use`/`tool_result` pair, is resent on every request.
- Confining an agent's file and code tools to a `tempfile` sandbox (with path-traversal checks and a subprocess timeout) is a minimal, effective safety boundary for exactly this kind of demo — the agent physically cannot touch the real repository.
- This raw loop is the same core mechanism that CLI coding agents like Claude Code are built on top of; what those products add on top is a much larger built-in toolset, permissions/approval flows, context compaction, and orchestration — not a different underlying protocol.
- This notebook is a deliberate, repo-documented exception to the `helpers.get_llm()` convention used elsewhere in this repository, because it specifically teaches first-party `anthropic` SDK usage rather than a LangChain-wrapped chat model.